In [1]:
# Install Hugging Face Transformers and Datasets
!pip install -q transformers timm

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from transformers import SwinForImageClassification, SwinConfig, AutoFeatureExtractor
import time
import numpy as np


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 111.7 MB/s eta 0:00:00


In [2]:
# Load the feature extractor for Swin
extractor = AutoFeatureExtractor.from_pretrained("microsoft/swin-tiny-patch4-window7-224")

# Transform: resize to 224x224 + normalization from extractor
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=extractor.image_mean, std=extractor.image_std)
])

# Datasets
train_dataset = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/vit/feature_extraction_vit.py:28: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(
100%|██████████| 169M/169M [00:12<00:00, 13.1MB/s]


In [3]:
####Fine-Tune Pretrained Swin Model
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x).logits
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total

def train_swin(model_name, epochs=3):
    print(f"\n🔧 Fine-tuning {model_name}...")
    model = SwinForImageClassification.from_pretrained(model_name)

    # Adjust classification head for 100 classes
    model.classifier = nn.Linear(model.classifier.in_features, 100)

    # Freeze backbone
    for param in model.swin.parameters():
        param.requires_grad = False

    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
    criterion = nn.CrossEntropyLoss()

    times = []
    for epoch in range(epochs):
        model.train()
        epoch_start = time.time()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x).logits
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
        epoch_time = time.time() - epoch_start
        times.append(epoch_time)
        print(f"Epoch {epoch+1}: Time = {epoch_time:.2f}s")

    acc = evaluate(model, test_loader)
    print(f"✅ Final Test Accuracy for {model_name}: {acc*100:.2f}%")
    return round(acc*100, 2), round(np.mean(times), 2)


In [4]:
####Run for Swin-Tiny and Swin-Small
results = []

# Fine-tune pretrained Swin-Tiny
acc_tiny, time_tiny = train_swin("microsoft/swin-tiny-patch4-window7-224", epochs=3)
results.append({
    "Model": "Swin-Tiny (Pretrained)",
    "Accuracy (%)": acc_tiny,
    "Epoch Time (s)": time_tiny
})

# Fine-tune pretrained Swin-Small
acc_small, time_small = train_swin("microsoft/swin-small-patch4-window7-224", epochs=3)
results.append({
    "Model": "Swin-Small (Pretrained)",
    "Accuracy (%)": acc_small,
    "Epoch Time (s)": time_small
})



🔧 Fine-tuning microsoft/swin-tiny-patch4-window7-224...


config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/113M [00:00<?, ?B/s]

Epoch 1: Time = 64.76s
Epoch 2: Time = 63.73s
Epoch 3: Time = 63.78s
✅ Final Test Accuracy for microsoft/swin-tiny-patch4-window7-224: 62.15%

🔧 Fine-tuning microsoft/swin-small-patch4-window7-224...


config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/199M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/199M [00:00<?, ?B/s]

Epoch 1: Time = 100.42s
Epoch 2: Time = 100.36s
Epoch 3: Time = 100.44s
✅ Final Test Accuracy for microsoft/swin-small-patch4-window7-224: 66.56%


In [5]:
#######Swin Transformer From Scratch

def train_swin_scratch():
    print("\n Training Swin-Tiny from scratch...")

    config = SwinConfig(image_size=224, num_labels=100)
    model = SwinForImageClassification(config)

    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
    criterion = nn.CrossEntropyLoss()

    times = []
    for epoch in range(3):
        model.train()
        epoch_start = time.time()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x).logits
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
        epoch_time = time.time() - epoch_start
        times.append(epoch_time)
        print(f"Epoch {epoch+1}: Time = {epoch_time:.2f}s")

    acc = evaluate(model, test_loader)
    print(f" Final Accuracy (Swin from scratch): {acc*100:.2f}%")
    return round(acc*100, 2), round(np.mean(times), 2)

acc_scratch, time_scratch = train_swin_scratch()
results.append({
    "Model": "Swin-Tiny (Scratch)",
    "Accuracy (%)": acc_scratch,
    "Epoch Time (s)": time_scratch
})



 Training Swin-Tiny from scratch...
Epoch 1: Time = 172.48s
Epoch 2: Time = 172.13s
Epoch 3: Time = 172.20s
 Final Accuracy (Swin from scratch): 29.18%


In [7]:
import pandas as pd
from IPython.display import display

df = pd.DataFrame(results)
display(df)


,Model,Accuracy (%),Epoch Time (s)
0,Swin-Tiny (Pretrained),62.15,64.09
1,Swin-Small (Pretrained),66.56,100.41
2,Swin-Tiny (Scratch),29.18,172.27
